In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm

In [ ]:
from band_plot_style import COLOR_DICT, MARK_DICT


# plotting original photometry

In [2]:
orig_phot = pd.read_csv('/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Inputs/Photometry/1_LCs_flux_raw/AT2017gfo.dat')


In [ ]:
orig_phot

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D

_DEFAULT_MARKERS = ['o', 's', '^', 'D', 'v', 'p', '*', 'h', 'X', 'P', '8']
_MARKER_FALLBACK = list(_DEFAULT_MARKERS)


def telescope_from_band(band):
    """Telescope name: part of `band` before first '_'; e.g. Sinistro_V → Sinistro."""
    if pd.isna(band):
        return 'Unknown'
    s = str(band).strip()
    if '_' in s:
        return s.split('_', 1)[0]
    return s


def band_color_template(df, band_col='band'):
    """
    Build a band -> color dict with None placeholders (matplotlib auto palette used until set).
    Keys are sorted unique band strings for stable copy-paste editing.
    """
    bands = sorted(df[band_col].dropna().unique(), key=str)
    return {b: None for b in bands}


def telescope_marker_template(df, band_col='band'):
    """
    Assign a distinct marker per unique telescope (band prefix before '_'), sorted, cycling markers.
    """
    tel = sorted(
        {telescope_from_band(b) for b in df[band_col].dropna().unique()},
        key=str,
    )
    return {t: _DEFAULT_MARKERS[i % len(_DEFAULT_MARKERS)] for i, t in enumerate(tel)}


def plot_magnitude_by_band(
    df,
    time_col='MJD',
    mag_col='Mag',
    band_col='band',
    err_col='Mag_err',
    color_dict=None,
    marker_dict=None,
    bands_to_plot=None,
    phase_offset=57982.52851852,
    legend_ncol=None,
    legend_max_ncol=6,
):
    """
    Magnitude vs time: color from band (filter), marker from telescope (band prefix before '_').

    Parameters
    ----------
    df : pd.DataFrame
    time_col, mag_col, band_col, err_col : str
        Column names for time, magnitude, band/filter, and magnitude error.
    color_dict : dict or None
        band string -> matplotlib color. None or missing keys fall back to the shuffled
        spectral palette; explicit None as a value also falls back to auto colors.
    marker_dict : dict or None
        telescope name (band prefix) -> marker. If None, uses telescope_marker_template(df).
        Unknown telescopes get a cyclic fallback marker.
    bands_to_plot : list or None
        If set, only these band names are drawn.
    phase_offset : float
        Subtracted from time_col to get phase (days).
    legend_ncol : int or None
        Matplotlib legend ncol (columns); if None, wraps using at most legend_max_ncol columns
        so the legend spans multiple rows when needed.
    legend_max_ncol : int
        When legend_ncol is None, ncol = min(legend_max_ncol, n_telescopes).
    """

    plt.rcParams['font.family'] = 'serif'
    fig, ax = plt.subplots(figsize=(14, 6))

    df_sorted = df.sort_values(by=time_col).copy()
    df_sorted[time_col] = pd.to_numeric(df_sorted[time_col], errors='coerce')
    df_sorted = df_sorted.dropna(subset=[time_col, mag_col])
    df_sorted['_tel'] = df_sorted[band_col].map(telescope_from_band)

    if bands_to_plot:
        df_sorted = df_sorted[df_sorted[band_col].isin(bands_to_plot)]

    if df_sorted.empty:
        plt.show()
        return

    unique_bands = df_sorted[band_col].unique()
    num_bands = len(unique_bands)
    auto_colors = cm.nipy_spectral(np.linspace(0, 1, max(num_bands, 1)))
    np.random.seed(42)
    np.random.shuffle(auto_colors)
    band_to_i = {b: i for i, b in enumerate(unique_bands)}

    def band_color(band):
        if color_dict and band in color_dict and color_dict[band] is not None:
            return color_dict[band]
        return auto_colors[band_to_i[band]]

    if marker_dict is None:
        marker_dict = telescope_marker_template(df_sorted, band_col)

    resolved_marker = dict(marker_dict)
    unique_tels = sorted(df_sorted['_tel'].unique(), key=str)
    fi = 0
    for t in unique_tels:
        if t not in resolved_marker:
            resolved_marker[t] = _MARKER_FALLBACK[fi % len(_MARKER_FALLBACK)]
            fi += 1

    for (band, tel), g in df_sorted.groupby([band_col, '_tel'], sort=False):
        color = band_color(band)
        mk = resolved_marker[tel]
        phase = g[time_col] - phase_offset
        ax.errorbar(
            phase,
            g[mag_col],
            yerr=g[err_col],
            color=color,
            alpha=0.8,
            fmt=mk,
            capsize=3,
            markeredgecolor='black',
            markerfacecolor=color,
            label='_nolegend_',
        )
    #ax.grid(True, which='both', axis='both', color='gray', alpha=0.5)
    ax.invert_yaxis()
    ax.set_xlim(0.3, 30)

    ax.tick_params(axis='both', which='major', labelsize=14)
    #ax.tick_params(axis='both', which='minor', labelsize=8)

    ax.set_xscale('log')
    ax.set_xlabel('Phase relative to merger (days)', fontsize=16)
    ax.set_ylabel('AB Magnitude', fontsize=16)

    n_tel = len(unique_tels)
    if legend_ncol is not None:
        ncol_leg = max(1, int(legend_ncol))
    else:
        ncol_leg = max(1, min(legend_max_ncol, n_tel))
    n_leg_rows = int(np.ceil(n_tel / ncol_leg)) if n_tel else 1

    leg_handles = [
        Line2D(
            [0],
            [0],
            linestyle='None',
            marker=resolved_marker[t],
            color='k',
            markerfacecolor='none',
            markeredgecolor='k',
            markersize=plt.rcParams.get('lines.markersize', 6.0),
        )
        for t in unique_tels
    ]
    ax.legend(
        leg_handles,
        unique_tels,
        loc='lower center',
        bbox_to_anchor=(0.5, 1.02),
        ncol=ncol_leg,
        frameon=False,
        fontsize=12,
    )

    top_margin = 0.90 - 0.042 * max(0, n_leg_rows - 1)
    top_margin = max(0.62, min(0.93, top_margin))
    fig.subplots_adjust(top=top_margin)
    plt.savefig('/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/input_photometry.png', dpi=500)
    plt.show()


In [ ]:
# Copy-paste templates (edit colors/markers, then pass into plot_magnitude_by_band):
# print(band_color_template(orig_phot))
# print(telescope_marker_template(orig_phot))

plot_magnitude_by_band(
    orig_phot,
    color_dict=COLOR_DICT,  # replace None with your colors per band
    marker_dict=MARKER_DICT,  # keys = band prefix before '_'; or telescope_marker_template(orig_phot)
)

In [ ]:
def plot_magnitude_by_suffix_groups(df, time_col='MJD', mag_col='Mag', band_col='band', err_col='Mag_err',
                                    color_dict=None):
    """
    Plots magnitude vs. time, creating a separate plot for each band suffix group,
    with a fixed x-axis across all plots for easy comparison.
    """
    # Ensure proper types and clean data
    df_sorted = df.sort_values(by=time_col).copy()
    df_sorted[time_col] = pd.to_numeric(df_sorted[time_col], errors='coerce')
    df_sorted = df_sorted.dropna(subset=[time_col, mag_col])

    # Calculate Phase for the entire DataFrame upfront
    df_sorted['Phase'] = df_sorted[time_col] - 57982.52851852

    # Extract the suffix to group by
    df_sorted['suffix'] = df_sorted[band_col].astype(str).str.split('_').str[-1].str.lower()

    # Determine global x-axis limits (filtering for Phase > 0 for the log scale)
    positive_phases = df_sorted[df_sorted['Phase'] > 0]['Phase']
    
    # We multiply by 0.9 and 1.1 to add a 10% visual padding so points 
    # on the very edges don't get cut off by the plot borders.
    global_xmin = positive_phases.min() * 0.9
    global_xmax = positive_phases.max() * 1.1

    # Get the unique suffixes to loop through
    unique_suffixes = df_sorted['suffix'].unique()

    # Loop through each suffix group and create a separate plot
    for suffix in unique_suffixes:
        
        suffix_df = df_sorted[df_sorted['suffix'] == suffix]
        unique_bands_in_plot = suffix_df[band_col].unique()
        
        plt.figure(figsize=(14, 6))

        num_bands = len(unique_bands_in_plot)
        auto_colors = cm.nipy_spectral(np.linspace(0, 1, num_bands))
        np.random.seed(42)
        np.random.shuffle(auto_colors)

        for i, band in enumerate(unique_bands_in_plot):
            band_df = suffix_df[suffix_df[band_col] == band]
            
            color = color_dict.get(band, auto_colors[i]) if color_dict else auto_colors[i]
            
            # Phase is already calculated in the DataFrame, so we just grab the column
            plt.errorbar(band_df['Phase'], band_df[mag_col], yerr=band_df[err_col],
                        label=band, color=color, alpha=0.8, fmt='o',
                        capsize=3,               
                        markeredgecolor='black', 
                        markerfacecolor=color)   

        plt.gca().invert_yaxis()
        plt.xscale('log')
        
        # Apply the fixed global x-axis limits to this plot
        plt.xlim(global_xmin, global_xmax)
        
        plt.xlabel('Phase from explosion (days)')
        plt.ylabel('Magnitude')
        plt.title(f'Magnitude vs. Time - Band Suffix: {suffix.upper()}')
        
        plt.legend(title='Filter', fontsize='small', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

## Stacked spectra (multi-epoch)

Loads `*.dat` / `*_mangled_spec.txt` from a chosen folder, offsets by **tier** (XSHOOTER UVB+VIS+NIR at one MJD share a tier). **Earliest phase is drawn lowest**; later phases stack **upward**. Spacing between stacked tiers is **`offset_step`** (multiply that number to spread or compress the stack). **Drawing order**: earliest tiers use a **higher z-order** so they appear **on top** where traces overlap (toggle with `zorder_early_on_top`). Phase uses merger MJD `PHASE_REF_MJD_SPECTRA` (same as photometry).

**Labels** default to black, **right-hand** edge of each trace, **above** a local continuum, with **`label_zorder`** (default 100000) so annotations sit **above all spectrum curves**. Tune `label_x_inset_frac`, `label_dy`, `label_continuum_*`, or **`label_adjustments`** (dict keyed by **filename**): `dx_A`, `dy`, absolute `x`/`y`, `ha`/`va`.

**Colours**: `cmap` + `cmap_limits`, or **`tier_colors`** dict / list.

Switch `SPECTRA_SOURCE` to `original`, `smoothed`, or `mangled`, or set `spec_dir` directly.

In [8]:
from pathlib import Path
import re
from collections import defaultdict
import matplotlib as mpl

SPEC_DIR_ORIGINAL = Path(
    '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Inputs/Spectroscopy/1_spec_original/AT2017gfo'
)
SPEC_DIR_SMOOTHED = Path(
    '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Inputs/Spectroscopy/2_spec_smoothed/AT2017gfo'
)
SPEC_DIR_MANGLED = Path(
    '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/mangled_spectra'
)

PHASE_REF_MJD_SPECTRA = 57982.52851852  # same reference as photometry phase

# --- mangled pipeline (mirrors GP2dim_utils) ---

def _mangled_wls_linear_angstrom(wls):
    w = np.asarray(wls, dtype=float)
    mx = np.nanmax(w)
    if np.isfinite(mx) and mx > 200.0:
        return w
    return np.power(10.0, np.clip(w, -50.0, 8.0))


def _mangled_flux_linear_from_log10(flux):
    f = np.asarray(flux, dtype=float)
    return np.power(10.0, np.clip(f, -350.0, 300.0))


def _parse_xshooter_phase_from_name(name: str, phase_ref: float):
    m = re.search(r'Phase\+([0-9.]+)d', name, re.I)
    if m:
        return float(m.group(1))
    m2 = re.search(r'MJD-([0-9.]+)', name)
    if m2:
        return float(m2.group(1)) - phase_ref
    return None


def parse_spectrum_meta(path: Path, phase_ref: float = PHASE_REF_MJD_SPECTRA):
    """Return dict with path, phase_days, mjd, arm, telescope_id, label; or None if unknown."""
    path = Path(path)
    name = path.name
    stem = path.stem

    if '_mangled_spec' in stem:
        m = re.match(r'^([0-9]+\.[0-9]+)_mangled_spec', stem)
        if not m:
            return None
        mjd = float(m.group(1))
        phase_days = mjd - phase_ref
        code = int(round(mjd * 1e6)) % 10
        if code in (1, 2, 3):
            arm = {1: 'NIR', 2: 'UVB', 3: 'VIS'}[code]
            tel = 'XSHOOTER'
            label = f'{phase_days:.2f} d, XSHOOTER {arm}'
        else:
            arm = None
            tel = 'LDSS3+Magellan'
            label = f'{phase_days:.2f} d, {tel}'
        return {
            'path': path,
            'phase_days': phase_days,
            'mjd': mjd,
            'arm': arm,
            'telescope_id': tel,
            'label': label,
            'kind': 'mangled',
        }

    if 'XSHOOTER' in name.upper():
        nu = name.upper()
        arm = None
        for pfx in ('UVB', 'VIS', 'NIR'):
            if nu.startswith(pfx + '_'):
                arm = pfx
                break
        ph = _parse_xshooter_phase_from_name(name, phase_ref)
        if ph is None:
            return None
        mjd_m = re.search(r'MJD-([0-9.]+)', name)
        mjd = float(mjd_m.group(1)) if mjd_m else None
        label = f'{ph:.2f} d, XSHOOTER {arm or "?"}'
        return {
            'path': path,
            'phase_days': ph,
            'mjd': mjd,
            'arm': arm,
            'telescope_id': 'XSHOOTER',
            'label': label,
            'kind': 'dat_xshooter',
        }

    m_mag = re.match(r'^([0-9]+\.[0-9]+)_([A-Za-z0-9-]+)_([A-Za-z0-9-]+)\.dat$', name)
    if m_mag:
        mjd = float(m_mag.group(1))
        inst = m_mag.group(2)
        site = m_mag.group(3)
        phase_days = mjd - phase_ref
        tel = f'{inst}+{site}'
        label = f'{phase_days:.2f} d, {tel}'
        return {
            'path': path,
            'phase_days': phase_days,
            'mjd': mjd,
            'arm': None,
            'telescope_id': tel,
            'label': label,
            'kind': 'dat_other',
        }

    return None


def stack_tier_key(meta: dict, epoch_decimals: int = 3):
    """XSHOOTER arms at same MJD (to 3 decimals) share a tier; other: one tier per file."""
    if meta.get('telescope_id') == 'XSHOOTER':
        mj = meta.get('mjd')
        if mj is not None:
            return ('XSHOOTER', round(float(mj), 3))
        return ('XSHOOTER', round(float(meta['phase_days']), epoch_decimals))
    ep = round(float(meta['phase_days']), epoch_decimals)
    return (meta['telescope_id'], ep, meta['path'].name)


def _read_engrave_dat(path: Path, flux_col_preference: str):
    with open(path, encoding='utf-8', errors='replace') as f:
        line0 = f.readline()
    if line0.lstrip().startswith('#'):
        colnames = [c for c in re.split(r'\s+', line0.lstrip('#').strip()) if c]
        df = pd.read_csv(path, sep=r'\s+', names=colnames, comment=None, skiprows=1, engine='python')
    else:
        df = pd.read_csv(path, sep=r'\s+', header=0, engine='python')
    lower = {str(c).lower(): c for c in df.columns}
    if 'wavelength' in lower:
        wl_col = lower['wavelength']
    elif 'wls' in lower:
        wl_col = lower['wls']
    else:
        wl_col = df.columns[0]
    if flux_col_preference in df.columns:
        flx_col = flux_col_preference
    else:
        flux_candidates = [c for c in df.columns if 'flux' in str(c).lower()]
        flx_col = flux_candidates[0] if flux_candidates else df.columns[1]
    wl = pd.to_numeric(df[wl_col], errors='coerce').to_numpy()
    flx = pd.to_numeric(df[flx_col], errors='coerce').to_numpy()
    return wl, flx


def load_spectrum_array(meta: dict, flux_column_xshooter: str = 'flux_tell_corrected'):
    path = meta['path']
    if meta.get('kind') == 'mangled':
        df = pd.read_csv(
            path,
            sep=r'\s+',
            comment='#',
            header=None,
            usecols=[0, 1],
            names=['wls', 'flux'],
            engine='python',
        )
        w = df['wls'].to_numpy(dtype=float)
        f_log = df['flux'].to_numpy(dtype=float)
        wl = _mangled_wls_linear_angstrom(w)
        flx = _mangled_flux_linear_from_log10(f_log)
        return wl, flx
    return _read_engrave_dat(path, flux_column_xshooter)


def _normalize_flux(wl, flx, wl_norm_range, normalize: bool):
    if not normalize:
        return flx
    w0, w1 = wl_norm_range
    mask = (wl >= w0) & (wl <= w1) & np.isfinite(flx)
    if np.sum(mask) < 5:
        sub = np.abs(flx[np.isfinite(flx)])
    else:
        sub = np.abs(flx[mask])
    if sub.size == 0:
        return flx
    denom = float(np.nanpercentile(sub, 95))
    if not np.isfinite(denom) or denom == 0:
        denom = 1.0
    return flx / denom




In [10]:
def plot_stacked_spectra(
    spec_dir,
    phase_ref: float = PHASE_REF_MJD_SPECTRA,
    epoch_decimals: int = 3,
    offset_step: float = 1.15,
    normalize: bool = True,
    wl_norm_range=(3500, 90000),
    wl_plot_range=None,
    flux_column_xshooter: str = 'flux_tell_corrected',
    figsize=(18, 12),
    cmap='viridis',
    cmap_limits=(0.12, 0.92),
    alternate_colors: bool = True,
    tier_colors=None,
    label_text_color='black',
    label_fontsize=8,
    label_x_inset_frac=0.02,
    label_continuum_right_frac=0.20,
    label_continuum_percentile=88.0,
    label_dy=0.08,
    label_adjustments=None,
    label_zorder=100000,
    zorder_early_on_top=True,
    zorder_base=2,
):
    """
    Stack normalized spectra. Earliest phase is lowest on the plot; later phases are higher.

    Vertical spacing between tiers is ``offset_step`` (same units as normalized flux after each
    spectrum is offset by ``offset_step * tier_index``). Increase it for more separation.

    Drawing order: if ``zorder_early_on_top``, **earliest** tiers get **higher** z-order so they
    are drawn on top where curves cross; **later** tiers sit underneath.

    Colors: ``cmap`` + ``cmap_limits`` (two floats in [0,1] sampled along the colormap), or
    pass ``tier_colors`` as a dict ``tier_tuple -> color`` or a list in chronological tier order.
    If ``alternate_colors=True``, it interleaves the colormap to maximize contrast between neighbors.

    Labels (black): anchored at the long-λ edge of each trace, sitting above an estimated
    continuum (percentile over the right ``label_continuum_right_frac`` of the trace).
    Text uses ``label_zorder`` (large default) so labels stay above **all** spectrum lines.

    Per-file overrides via ``label_adjustments`` keyed by filename; see markdown above demo cell.
    """
    spec_dir = Path(spec_dir)
    paths = sorted(spec_dir.glob('*.dat')) + sorted(spec_dir.glob('*_mangled_spec.txt'))
    records = []
    for p in paths:
        meta = parse_spectrum_meta(p, phase_ref)
        if meta is None:
            continue
        meta['tier'] = stack_tier_key(meta, epoch_decimals)
        try:
            wl, flx = load_spectrum_array(meta, flux_column_xshooter)
        except Exception:
            continue
        meta['wl'] = wl
        meta['flx'] = _normalize_flux(wl, flx, wl_norm_range, normalize)
        records.append(meta)

    if not records:
        print('No spectra loaded from', spec_dir)
        return

    by_tier = defaultdict(list)
    for r in records:
        by_tier[r['tier']].append(r)

    # Sort everything by time
    tier_order = sorted(by_tier.keys(), key=lambda t: (float(t[1]), str(t[-1]) if len(t) > 2 else ''))
    n_tier = len(tier_order)
    lo, hi = cmap_limits
    
    if tier_colors is None:
        cmap_obj = mpl.colormaps[cmap] if isinstance(cmap, str) else cmap
        colors = cmap_obj(np.linspace(lo, hi, max(n_tier, 1)))
        
        # Interleave colormap to increase adjacent contrast
        if alternate_colors and n_tier > 1:
            half = (n_tier + 1) // 2
            interleaved = []
            for idx in range(half):
                interleaved.append(colors[idx])
                if idx + half < n_tier:
                    interleaved.append(colors[idx + half])
            colors = interleaved
            
        tier_color = {t: colors[i] for i, t in enumerate(tier_order)}
    elif isinstance(tier_colors, dict):
        cmap_obj = mpl.colormaps[cmap] if isinstance(cmap, str) else cmap
        fallback = cmap_obj(np.linspace(lo, hi, max(n_tier, 1)))
        tier_color = {t: tier_colors.get(t, fallback[i]) for i, t in enumerate(tier_order)}
    else:
        tc = list(tier_colors)
        if len(tc) < n_tier:
            raise ValueError(f'tier_colors has {len(tc)} entries but there are {n_tier} tiers')
        tier_color = {t: tc[i] for i, t in enumerate(tier_order)}

    label_adjustments = label_adjustments or {}
    label_dy_base = label_dy * offset_step
    plt.rcParams['font.family'] = 'serif'
    fig, ax = plt.subplots(figsize=figsize)
    global_wl_min, global_wl_max = np.inf, -np.inf
    ax.axvspan(13125, 14375, color='lightgray', alpha=0.5)
    ax.axvspan(17812.5, 19375, color='lightgray', alpha=0.5)
    ax.axvspan(9800, 10100, color='lightgray', alpha=0.5)
    ax.axvspan(5300, 5700, color='lightgray', alpha=0.5)
    

    for i, tier in enumerate(tier_order):
        y_off = offset_step * i
        c = tier_color[tier]
        if zorder_early_on_top:
            z_line = zorder_base + (n_tier - 1 - i)
        else:
            z_line = zorder_base + i
        for j, r in enumerate(by_tier[tier]):
            wl, y = r['wl'], np.asarray(r['flx'], dtype=float) + y_off
            m = np.isfinite(wl) & np.isfinite(y)
            wl, y = wl[m], y[m]
            if wl.size < 2:
                continue
            ax.plot(wl, y, color=c, lw=0.9, alpha=0.95, zorder=z_line + 1e-3 * j)
            global_wl_min = min(global_wl_min, float(np.nanmin(wl)))
            global_wl_max = max(global_wl_max, float(np.nanmax(wl)))

            fname = r['path'].name
            adj = label_adjustments.get(fname, {})

            x_lo = float(np.nanmin(wl))
            x_hi = float(np.nanmax(wl))
            span = max(x_hi - x_lo, 1.0)
            if 'x' in adj:
                x_text = float(adj['x'])
            else:
                x_text = x_hi - label_x_inset_frac * span + float(adj.get('dx_A', 0.0))

            right_lo = x_hi - label_continuum_right_frac * span
            sel = wl >= right_lo
            if not np.any(sel):
                sel = np.ones_like(wl, dtype=bool)
            y_ref = float(np.nanpercentile(y[sel], label_continuum_percentile))
            if 'y' in adj:
                y_text = float(adj['y'])
            else:
                y_text = y_ref + label_dy_base + float(adj.get('dy', 0.0))

            ha = adj.get('ha', 'right')
            va = adj.get('va', 'bottom')

            ax.text(
                x_text,
                y_text,
                r['label'],
                color=label_text_color,
                fontsize=9,
                horizontalalignment=ha,
                verticalalignment=va,
                clip_on=True,
                zorder=label_zorder + i + 1e-3 * j,
                # Here is the new bounding box modification!
                bbox=dict(
                    facecolor='white', 
                    alpha=0.7, 
                    edgecolor='none', 
                    boxstyle='round,pad=0.3'
                )
            )

    ax.set_xlabel(r'Rest wavelength ($\mathrm{\AA}$)', fontsize=16)
    ax.set_ylabel('Normalized flux + offset' if normalize else r'Flux + offset', fontsize=16)
    ax.tick_params(axis='both', which='major', labelsize=14)
    if wl_plot_range is not None:
        ax.set_xlim(wl_plot_range)
    else:
        ax.set_xlim(global_wl_min, global_wl_max)
    ax.set_ylim(-1,25)
    plt.tight_layout()
    plt.savefig('/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/smoothed_spectra_legible.png', dpi=600)
    plt.show()


In [ ]:
# Choose which spectrum set to plot: "original" | "smoothed" | "mangled"
SPECTRA_SOURCE = 'smoothed'

spec_dir_map = {
    'original': SPEC_DIR_ORIGINAL,
    'smoothed': SPEC_DIR_SMOOTHED,
    'mangled': SPEC_DIR_MANGLED,
}
spec_dir = spec_dir_map[SPECTRA_SOURCE]

# Optional: nudge individual labels (keys = exact filename under spec_dir)
LABEL_ADJ = {
    '57983.0362_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57983.01825_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57983.02788_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57983.05998_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57983.9914_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57984.0484_LDSS3_Magellan.dat': {'dy': -0.2, 'dx_A': 2800},
    '57985.99019_LDSS3_Magellan.dat': {'dy': -0.4, 'dx_A': 2400},
    '57987.03511_LDSS3_Magellan.dat': {'dy': -0.8, 'dx_A': 2300},
    '57989.98185_LDSS3_Magellan.dat': {'dy': -2.5, 'dx_A': 1800},
    '57990.98563_LDSS3_Magellan.dat': {'dy': -3.5, 'dx_A': 1800},
    # 'UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969002_Phase+1.43d.dat': {'dy': 0.05, 'dx_A': -80},
    # '57983.0362_LDSS3_Magellan.dat': {'x': 9800, 'y': 14.2},
}


plot_stacked_spectra(
    spec_dir,
    wl_plot_range=(3000, 25000),
    wl_norm_range=(3800, 9000),
    cmap='spring',
    cmap_limits=(0.15, 0.95),
    label_adjustments=LABEL_ADJ,
    label_dy=0.10,
    offset_step=1.15,  # increase to separate stacked tiers vertically
    zorder_early_on_top=True,
    alternate_colors= True  # earliest phase drawn on top where curves cross
)

## FINAL spectra: smooth SED surface (log-scaled flux)

- **Data:** `*_FINAL*.txt` under `FINAL_spectra_2dim/...` (toggle RJF branch in the code cell). Not the GP grid from notebook 6.
- **Plot:** Irregular `(phase, λ)` samples are interpolated onto a regular grid with SciPy **`griddata`**, then drawn as **`pcolormesh`** or **`contourf`**. The colormap uses **log10 F_λ** (observer or rest if `APPLY_REDSHIFT`); the colorbar tick labels show **F_λ** as powers of ten (e.g. \(10^{-16}\)). Regions outside the data convex hull are masked (NaN).
- **Optional:** Gaussian blur (`GAUSSIAN_SMOOTH_SIGMA`) in grid pixels for a smoother look (visualization only).
- **Stems:** Log-phase filenames need `Outputs/<SN>/fitted_phot_logspace_<SN>.dat` for `stem_to_spec_mjd` (same as notebook 7.5).


In [ ]:
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime
rt = pconf.bootstrap_runtime()
FINAL_SPECTRA_DIR = rt.final_spectra_dir
# Visualization: linear/cubic interpolation + log10(F_λ) on a grid — not physical RT.
# "cubic" can overshoot; "linear" is safer. Gaps leave masked holes, not invented flux.
import os
import sys
import glob
import warnings

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

APPLY_REDSHIFT = True  # λ_rest = λ/(1+z), F_λ,rest = F_λ,obs * (1+z); ignored if Z <= 0

FINAL_SUFFIXES = None  # e.g. ("_FINAL_spec.txt",) or None for all *_FINAL*.txt
FINAL_FLUX_ON_DISK = "auto"

AXES_MODE = "linear_linear"  # "linear_linear" | "log_log"
PHASE_EPS_DAYS = 1e-4

# Interpolated surface
GRID_N_PHASE = 200
GRID_N_WL = 200
INTERP_METHOD = "cubic"  # "linear" | "cubic"
FLUX_FLOOR = 1e-40  # floor inside log10(F_λ)
GAUSSIAN_SMOOTH_SIGMA = 2  # 0 = off; or scalar; or (sigma_phase, sigma_wl) in grid pixels
PLOT_STYLE = "pcolormesh"  # "pcolormesh" | "contourf"
CONTOURF_LEVELS = 64

CMAP = "viridis"
VMIN = None
VMAX = None
FIGSIZE = (10, 4)

_CODES = os.path.join(os.path.normpath(COCO_PATH.rstrip(os.sep)), "Codes")
if _CODES not in sys.path:
    sys.path.insert(0, _CODES)


from comparison_check_log_utils import (
    resolve_final_directory,
    read_final_spectrum_linear,
    parse_final_stem,
    stem_to_spec_mjd,
)

if TWODIM_MODE_SHORT is None:
    if TWODIM_PRODUCT is None:

if INTERP_METHOD not in ("linear", "cubic"):
    raise ValueError("INTERP_METHOD must be linear or cubic")
if PLOT_STYLE not in ("pcolormesh", "contourf"):
    raise ValueError("PLOT_STYLE must be pcolormesh or contourf")

T0_MJD = float(pconf.SN_EXPLOSION_MJD.get(SNNAME, 57982.52851852))

_branch = pconf.final_spectra_twodim_branch(TWODIM_MODE_SHORT, TWODIM_PRODUCT,
    use_iter_gp_mangle=True)
FINAL_DIR = resolve_final_directory(
    COCO_PATH, SNNAME, FINAL_VARIANT, twodim_branch=_branch
)
print("FINAL_DIR:", FINAL_DIR)

pat = os.path.join(FINAL_DIR, "*_FINAL*.txt")
paths = sorted(glob.glob(pat))
if FINAL_SUFFIXES:
    paths = [p for p in paths if any(p.endswith(s) for s in FINAL_SUFFIXES)]

ph_list, wl_list, fl_list = [], [], []
n_skip = 0
n_ok = 0

for _path in paths:
    _base = os.path.basename(_path)
    try:
        stem = parse_final_stem(_base)
        mjd = stem_to_spec_mjd(
            stem, COCO_PATH, SNNAME, datalc_path=DATALC_PATH
        )
    except Exception as e:
        warnings.warn("%s: %s" % (_base, e))
        n_skip += 1
        continue
    phase_days = float(mjd - T0_MJD)
    try:
        wl, flux, _fe = read_final_spectrum_linear(
            _path, flux_on_disk=FINAL_FLUX_ON_DISK
        )
    except Exception as e:
        warnings.warn("%s read: %s" % (_base, e))
        n_skip += 1
        continue
    wl = np.asarray(wl, dtype=float)
    flux_lin = np.asarray(flux, dtype=float)
    if APPLY_REDSHIFT and Z > 0:
        oz = 1.0 + float(Z)
        wl = wl / oz
        flux_lin = flux_lin * oz
    m = np.isfinite(wl) & np.isfinite(flux_lin) & (flux_lin > 0)
    wl, flux_lin = wl[m], flux_lin[m]
    if wl.size == 0:
        n_skip += 1
        continue
    ph = np.full_like(wl, phase_days, dtype=float)
    ph_list.append(ph)
    wl_list.append(wl)
    fl_list.append(flux_lin)
    n_ok += 1

if not ph_list:
    raise RuntimeError("No spectra loaded (ok=%i skip=%i)." % (n_ok, n_skip))

ph_a = np.concatenate(ph_list)
wl_a = np.concatenate(wl_list)
flux_a = np.concatenate(fl_list)
z_a = np.log10(np.maximum(flux_a, float(FLUX_FLOOR)))

if AXES_MODE == "linear_linear":
    xp, yp = ph_a, wl_a
    xlabel = "Phase relative to merger (days)"
    ylabel = "Wavelength (Å)"
    xi = np.linspace(float(np.min(xp)), float(np.max(xp)), int(GRID_N_PHASE))
    yi = np.linspace(float(np.min(yp)), float(np.max(yp)), int(GRID_N_WL))
elif AXES_MODE == "log_log":
    xp = np.log10(np.maximum(ph_a, float(PHASE_EPS_DAYS)))
    yp = np.log10(np.clip(wl_a, 1e-300, np.inf))
    xlabel = "log10 phase (d)"
    ylabel = "log10 λ (Å)"
    xi = np.linspace(float(np.min(xp)), float(np.max(xp)), int(GRID_N_PHASE))
    yi = np.linspace(float(np.min(yp)), float(np.max(yp)), int(GRID_N_WL))
else:
    raise ValueError("AXES_MODE must be linear_linear or log_log")

Xi, Yi = np.meshgrid(xi, yi)
pts = np.column_stack([xp.astype(float), yp.astype(float)])
Zi = griddata(pts, z_a.astype(float), (Xi, Yi), method=INTERP_METHOD)

if GAUSSIAN_SMOOTH_SIGMA not in (0, None):
    sig = GAUSSIAN_SMOOTH_SIGMA
    if np.isscalar(sig):
        sig = (float(sig), float(sig))
    else:
        sig = (float(sig[0]), float(sig[1]))
    Zv = np.asarray(Zi, dtype=float)
    mask = np.isfinite(Zv)
    if mask.any():
        num = np.where(mask, Zv, 0.0)
        den = np.where(mask, 1.0, 0.0)
        num = gaussian_filter(num, sigma=sig)
        den = gaussian_filter(den, sigma=sig)
        Zi = np.where(den > 1e-9, num / den, np.nan)

Zi_ma = np.ma.masked_invalid(Zi)


def _flux_cbar_tickfmt(log10_f, _pos):
    # Mesh/cmap use log10(F_λ); ticks show linear flux as powers of ten.
    if not np.isfinite(log10_f):
        return ""
    if abs(log10_f - round(log10_f)) < 1e-5:
        e = int(round(log10_f))
        return r"$10^{%d}$" % e
    return r"$10^{%.1f}$" % log10_f

plt.rcParams['font.family'] = 'serif'
fig, ax = plt.subplots(figsize=FIGSIZE)
if PLOT_STYLE == "pcolormesh":
    pcm = ax.pcolormesh(
        Xi,
        Yi,
        Zi_ma,
        shading="auto",
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
    )
else:
    pcm = ax.contourf(
        Xi,
        Yi,
        Zi_ma.filled(np.nan),
        levels=int(CONTOURF_LEVELS),
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        extend="both",
    )

cb = fig.colorbar(pcm, ax=ax)

# Force the major ticks to land on integers 
cb.ax.yaxis.set_major_locator(MultipleLocator(base=1.0))
cb.ax.yaxis.set_major_formatter(FuncFormatter(_flux_cbar_tickfmt))
cb.ax.tick_params(labelsize=14)

cb.set_label(
    r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)" if APPLY_REDSHIFT and Z > 0 else "", fontsize=16
)
ax.tick_params(axis='both', which='major', labelsize=14)
ax.set_xlabel(xlabel, fontsize=16)
ax.set_ylabel(ylabel, fontsize=16)
# ax.set_title(
#     "%s FINAL SED (grid %s, %i epochs, %i skips)"
#     % (SNNAME, INTERP_METHOD, n_ok, n_skip)
# )
plt.tight_layout()
plt.savefig('/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/final_sed_heatmap_ryanversion.png', dpi=600, bbox_inches='tight')
plt.show()
print(
    "Loaded epochs: %i | skipped: %i | branch: %s | grid %ix%i"
    % (n_ok, n_skip, _branch, GRID_N_PHASE, GRID_N_WL)
)


## FINAL SED: slice at fixed wavelength (presentation)

- **Purpose:** Cut the **same** 2D interpolated surface as the heatmap (`griddata` on log₁₀ F_λ, same `AXES_MODE` and hull). Useful for slides so the line matches the `pcolormesh`.
- **Input:** `WL_TARGET_AA` in **Å**, in the **same frame** as the heatmap (**rest** if `APPLY_REDSHIFT` is True).
- **`SLICE_SOURCE`:** `"grid"` (default) = slice the grid; `"native"` = per-epoch linear `np.interp` in λ from each FINAL file (can disagree with the surface; us e if you want bin-style sampling).
- **Smoother look:** Turn up **`GAUSSIAN_SMOOTH_SIGMA`** (2D, before slicing), use **`INTERP_METHOD = "cubic"`**, denser **`GRID_N_*`**, and/or **`SLICE_PHASE_SMOOTH_SIGMA`** (1D Gaussian along phase on the slice only — **visualization only**).
- **Contrast with notebook 7.5:** There, `plot_lightcurve` takes the **nearest column** on a lookup table with a fixed λ grid. Here the default follows **this** notebook’s interpolated FINAL map.


In [ ]:
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime
rt = pconf.bootstrap_runtime()
FINAL_SPECTRA_DIR = rt.final_spectra_dir
# Grid slice matches the pcolormesh surface. Native mode samples each 1D spectrum at lambda.
import os
import sys
import glob
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter, gaussian_filter1d

APPLY_REDSHIFT = True

FINAL_SUFFIXES = None
FINAL_FLUX_ON_DISK = "auto"

AXES_MODE = "linear_linear"  # "linear_linear" | "log_log"
PHASE_EPS_DAYS = 1e-4

GRID_N_PHASE = 200
GRID_N_WL = 200
INTERP_METHOD = "cubic"
FLUX_FLOOR = 1e-40
GAUSSIAN_SMOOTH_SIGMA = 1

# Slice-specific
WL_TARGET_AA = 15000.0  # Angstrom; rest-frame if APPLY_REDSHIFT
SLICE_SOURCE = "grid"  # "grid" | "native"
LC_X_AXIS = "phase"  # "phase" | "mjd"
SLICE_PHASE_SMOOTH_SIGMA = 0.0  # 1D Gaussian on slice (log10 flux); viz only; 0 = off
LOG_FLUX_Y = False

LC_FIGSIZE = (6, 4)

_CODES = os.path.join(os.path.normpath(COCO_PATH.rstrip(os.sep)), "Codes")
if _CODES not in sys.path:
    sys.path.insert(0, _CODES)


from comparison_check_log_utils import (
    resolve_final_directory,
    read_final_spectrum_linear,
    parse_final_stem,
    stem_to_spec_mjd,
)

if TWODIM_MODE_SHORT is None:
    if TWODIM_PRODUCT is None:

if INTERP_METHOD not in ("linear", "cubic"):
    raise ValueError("INTERP_METHOD must be linear or cubic")
if SLICE_SOURCE not in ("grid", "native"):
    raise ValueError("SLICE_SOURCE must be grid or native")
if LC_X_AXIS not in ("phase", "mjd"):
    raise ValueError("LC_X_AXIS must be phase or mjd")

T0_MJD = float(pconf.SN_EXPLOSION_MJD.get(SNNAME, 57982.52851852))

_branch = pconf.final_spectra_twodim_branch(TWODIM_MODE_SHORT, TWODIM_PRODUCT,
    use_iter_gp_mangle=True)
FINAL_DIR = resolve_final_directory(
    COCO_PATH, SNNAME, FINAL_VARIANT, twodim_branch=_branch
)

pat = os.path.join(FINAL_DIR, "*_FINAL*.txt")
paths = sorted(glob.glob(pat))
if FINAL_SUFFIXES:
    paths = [p for p in paths if any(p.endswith(s) for s in FINAL_SUFFIXES)]

ph_list, wl_list, fl_list = [], [], []
n_skip = 0
n_ok = 0

for _path in paths:
    _base = os.path.basename(_path)
    try:
        stem = parse_final_stem(_base)
        mjd = stem_to_spec_mjd(
            stem, COCO_PATH, SNNAME, datalc_path=DATALC_PATH
        )
    except Exception as e:
        warnings.warn("%s: %s" % (_base, e))
        n_skip += 1
        continue
    phase_days = float(mjd - T0_MJD)
    try:
        wl, flux, _fe = read_final_spectrum_linear(
            _path, flux_on_disk=FINAL_FLUX_ON_DISK
        )
    except Exception as e:
        warnings.warn("%s read: %s" % (_base, e))
        n_skip += 1
        continue
    wl = np.asarray(wl, dtype=float)
    flux_lin = np.asarray(flux, dtype=float)
    if APPLY_REDSHIFT and Z > 0:
        oz = 1.0 + float(Z)
        wl = wl / oz
        flux_lin = flux_lin * oz
    m = np.isfinite(wl) & np.isfinite(flux_lin) & (flux_lin > 0)
    wl, flux_lin = wl[m], flux_lin[m]
    if wl.size == 0:
        n_skip += 1
        continue
    ph = np.full_like(wl, phase_days, dtype=float)
    ph_list.append(ph)
    wl_list.append(wl)
    fl_list.append(flux_lin)
    n_ok += 1

if not ph_list:
    raise RuntimeError("No spectra loaded (ok=%i skip=%i)." % (n_ok, n_skip))

ph_a = np.concatenate(ph_list)
wl_a = np.concatenate(wl_list)
flux_a = np.concatenate(fl_list)
z_a = np.log10(np.maximum(flux_a, float(FLUX_FLOOR)))

if AXES_MODE == "linear_linear":
    xp, yp = ph_a, wl_a
    xi = np.linspace(float(np.min(xp)), float(np.max(xp)), int(GRID_N_PHASE))
    yi = np.linspace(float(np.min(yp)), float(np.max(yp)), int(GRID_N_WL))
elif AXES_MODE == "log_log":
    xp = np.log10(np.maximum(ph_a, float(PHASE_EPS_DAYS)))
    yp = np.log10(np.clip(wl_a, 1e-300, np.inf))
    xi = np.linspace(float(np.min(xp)), float(np.max(xp)), int(GRID_N_PHASE))
    yi = np.linspace(float(np.min(yp)), float(np.max(yp)), int(GRID_N_WL))
else:
    raise ValueError("AXES_MODE must be linear_linear or log_log")

Xi, Yi = np.meshgrid(xi, yi)
pts = np.column_stack([xp.astype(float), yp.astype(float)])


def _smooth_1d_phase(z1d, sigma):
    z1d = np.asarray(z1d, dtype=float)
    if float(sigma) <= 0 or not np.isfinite(sigma):
        return z1d
    m = np.isfinite(z1d)
    if not m.any():
        return z1d
    num = np.where(m, z1d, 0.0)
    den = np.where(m, 1.0, 0.0)
    num = gaussian_filter1d(num, float(sigma), mode="nearest")
    den = gaussian_filter1d(den, float(sigma), mode="nearest")
    return np.where(den > 1e-9, num / den, np.nan)


mjd_line = None

if SLICE_SOURCE == "grid":
    if AXES_MODE == "linear_linear":
        y_query = float(WL_TARGET_AA)
    else:
        y_query = float(np.log10(float(WL_TARGET_AA)))
    if not (yi.min() <= y_query <= yi.max()):
        raise ValueError(
            "WL_TARGET_AA maps to y_query=%.6g outside grid [%.6g, %.6g]"
            % (y_query, yi.min(), yi.max())
        )

    Zi = griddata(pts, z_a.astype(float), (Xi, Yi), method=INTERP_METHOD)

    if GAUSSIAN_SMOOTH_SIGMA not in (0, None):
        sig = GAUSSIAN_SMOOTH_SIGMA
        if np.isscalar(sig):
            sig = (float(sig), float(sig))
        else:
            sig = (float(sig[0]), float(sig[1]))
        Zv = np.asarray(Zi, dtype=float)
        mask = np.isfinite(Zv)
        if mask.any():
            num = np.where(mask, Zv, 0.0)
            den = np.where(mask, 1.0, 0.0)
            num = gaussian_filter(num, sigma=sig)
            den = gaussian_filter(den, sigma=sig)
            Zi = np.where(den > 1e-9, num / den, np.nan)

    # Zi[i,j] is (xi[j], yi[i]) with np.meshgrid(xi, yi)
    z_line = np.array(
        [np.interp(y_query, yi, Zi[:, j]) for j in range(Zi.shape[1])]
    )
    z_line = _smooth_1d_phase(z_line, SLICE_PHASE_SMOOTH_SIGMA)
    flux_line = 10.0 ** z_line
    x_coord = xi.copy()

elif SLICE_SOURCE == "native":
    x_native, z_native, m_native = [], [], []
    for _path in paths:
        _base = os.path.basename(_path)
        try:
            stem = parse_final_stem(_base)
            mjd = stem_to_spec_mjd(
                stem, COCO_PATH, SNNAME, datalc_path=DATALC_PATH
            )
        except Exception:
            continue
        phase_days = float(mjd - T0_MJD)
        try:
            wl, flux, _fe = read_final_spectrum_linear(
                _path, flux_on_disk=FINAL_FLUX_ON_DISK
            )
        except Exception:
            continue
        wl = np.asarray(wl, dtype=float)
        flux_lin = np.asarray(flux, dtype=float)
        if APPLY_REDSHIFT and Z > 0:
            oz = 1.0 + float(Z)
            wl = wl / oz
            flux_lin = flux_lin * oz
        m = np.isfinite(wl) & np.isfinite(flux_lin)
        wl, flux_lin = wl[m], flux_lin[m]
        if wl.size == 0:
            continue
        order = np.argsort(wl)
        wl, flux_lin = wl[order], flux_lin[order]
        wlo, whi = float(wl.min()), float(wl.max())
        if not (wlo <= float(WL_TARGET_AA) <= whi):
            continue
        f0 = float(np.interp(float(WL_TARGET_AA), wl, flux_lin))
        if f0 <= 0:
            continue
        x_native.append(phase_days)
        z_native.append(np.log10(max(f0, float(FLUX_FLOOR))))
        m_native.append(float(mjd))
    if not x_native:
        raise RuntimeError(
            "Native slice: no epochs bracket WL_TARGET_AA with positive flux."
        )
    x_native = np.asarray(x_native)
    z_native = np.asarray(z_native)
    order = np.argsort(x_native)
    x_native = x_native[order]
    z_native = z_native[order]
    mjd_line = np.asarray(m_native, dtype=float)[order]
    z_line = _smooth_1d_phase(z_native, SLICE_PHASE_SMOOTH_SIGMA)
    flux_line = 10.0 ** z_line
    x_coord = x_native
else:
    raise ValueError(SLICE_SOURCE)

if SLICE_SOURCE == "grid":
    if LC_X_AXIS == "phase":
        if AXES_MODE == "linear_linear":
            x_plot = x_coord
            xlab = "Phase relative to merger (days)"
        else:
            x_plot = 10.0 ** x_coord
            xlab = "Phase (days)"
    else:
        if AXES_MODE == "linear_linear":
            x_plot = x_coord + T0_MJD
            xlab = "MJD"
        else:
            x_plot = T0_MJD + 10.0 ** x_coord
            xlab = "MJD"
else:
    if LC_X_AXIS == "phase":
        x_plot = x_coord
        xlab = "Phase relative to merger (days)"
    else:
        x_plot = mjd_line
        xlab = "MJD"

fig, ax = plt.subplots(figsize=LC_FIGSIZE)
valid = np.isfinite(flux_line) & np.isfinite(x_plot)
ax.plot(x_plot[valid], flux_line[valid], "-", lw=2.5, color="deepskyblue", label='%.0f Å' % WL_TARGET_AA)
if LOG_FLUX_Y:
    ax.set_yscale("log")

frame_note = " (rest)" if (APPLY_REDSHIFT and Z > 0) else ""
wl_nm = float(WL_TARGET_AA) / 10.0
ax.set_xlabel(xlab, fontsize=16)
ax.set_ylabel(
    r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)", fontsize=16
)
# ax.set_title(
#     "%s -- %s slice at %.0f A (~%.1f nm)%s"
#     % (SNNAME, SLICE_SOURCE, WL_TARGET_AA, wl_nm, frame_note),
#     fontsize=12,
# )
ax.minorticks_on()

# 2. Customize the grid lines to show both major and minor grids
ax.grid(True, which='major', linestyle='-', linewidth=0.8, alpha=0.7)
ax.grid(True, which='minor', linestyle=':', linewidth=0.5, alpha=0.5)

# 3. Increase the font size of the tick labels (adjust '12' to your preference)
ax.tick_params(axis='both', which='major', labelsize=16)
ax.tick_params(axis='both', which='minor')
ax.legend(fontsize=16)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig('/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/final_sed_wavelength_slice.png', dpi=600, bbox_inches='tight')
plt.show()

print(
    "Slice: %s | wl = %.4g A%s | valid pts %i/%i | branch %s | epochs loaded %i"
    % (
        SLICE_SOURCE,
        WL_TARGET_AA,
        frame_note.strip(),
        int(np.sum(valid)),
        x_plot.size,
        _branch,
        n_ok,
    )
)
